# Duplicate Analysis: Current Combined Dataset

This notebook is an investigation only. It reads the current combined dataset and performs all duplicate analysis in memory. It does not modify, delete, move, rename, overwrite, regenerate, or export any dataset or metadata files.

The analysis uses exact SHA-256 hashes of the final PNG files. Duplicate pair counts are not treated as counts of unique duplicated images.

## 1. Imports and read-only paths

In [17]:
from collections import defaultdict
from hashlib import sha256
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

COMBINED_ROOT = PROJECT_ROOT / "data" / "processed" / "combined"
SPLITS = ("train", "valid", "test")
METADATA_COLUMNS = {"filename", "cancer", "dataset", "split"}

print(f"Project root: {PROJECT_ROOT}")
print(f"Combined dataset: {COMBINED_ROOT}")
print("Read-only analysis: no files will be written.")

Project root: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection
Combined dataset: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed\combined
Read-only analysis: no files will be written.


## 2. Load and validate combined metadata

In [10]:
metadata_by_split = {}
metadata_frames = []

for split in SPLITS:
    metadata_path = COMBINED_ROOT / split / "metadata.csv"
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Missing metadata file: {metadata_path}")

    metadata = pd.read_csv(metadata_path)
    missing_columns = METADATA_COLUMNS - set(metadata.columns)
    if missing_columns:
        raise ValueError(f"{metadata_path} is missing columns: {sorted(missing_columns)}")

    metadata = metadata.copy()
    metadata["split_from_path"] = split
    metadata_by_split[split] = metadata
    metadata_frames.append(metadata)

combined_metadata = pd.concat(metadata_frames, ignore_index=True)

print("Metadata columns by split:")
for split, metadata in metadata_by_split.items():
    print(f"{split}: {list(metadata.columns)} | rows={len(metadata):,}")

print("\nDataset values:")
print(combined_metadata["dataset"].value_counts(dropna=False))

print("\nDataset/split counts:")
display(pd.crosstab(combined_metadata["dataset"], combined_metadata["split_from_path"]))

Metadata columns by split:
train: ['filename', 'cancer', 'dataset', 'split', 'split_from_path'] | rows=10,052
valid: ['filename', 'cancer', 'dataset', 'split', 'split_from_path'] | rows=1,257
test: ['filename', 'cancer', 'dataset', 'split', 'split_from_path'] | rows=1,247

Dataset values:
dataset
Dataset2    8810
BTXRD       3746
Name: count, dtype: int64

Dataset/split counts:


split_from_path,test,train,valid
dataset,,,
BTXRD,375,2996,375
Dataset2,872,7056,882


## 3. Build exact SHA-256 records without changing files

Only final PNG files referenced by the combined metadata are hashed. The metadata split is checked against the directory split so that a duplicate cannot be assigned to the wrong split by accident.

In [11]:
def sha256_file(path):
    digest = sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

records = []
missing_images = []
non_png_images = []

for split, metadata in metadata_by_split.items():
    image_dir = COMBINED_ROOT / split / "images"
    for row in metadata.itertuples(index=False):
        image_path = image_dir / row.filename
        if not image_path.is_file():
            missing_images.append(str(image_path))
            continue
        if image_path.suffix.lower() != ".png":
            non_png_images.append(str(image_path))
        records.append({
            "hash": sha256_file(image_path),
            "filename": row.filename,
            "dataset": row.dataset,
            "split": split,
            "cancer": int(row.cancer),
            "path": str(image_path),
        })

hash_records = pd.DataFrame(records)
if missing_images:
    raise FileNotFoundError(f"Metadata references {len(missing_images)} missing image(s); first: {missing_images[:5]}")
if non_png_images:
    raise ValueError(f"Expected final PNG files, found {len(non_png_images)} non-PNG file(s); first: {non_png_images[:5]}")

print(f"Hashed image records: {len(hash_records):,}")
print(f"Unique file hashes: {hash_records['hash'].nunique():,}")
print(f"Dataset 2 records: {(hash_records['dataset'] == 'Dataset2').sum():,}")
print(f"BTXRD records: {(hash_records['dataset'] == 'BTXRD').sum():,}")

Hashed image records: 12,556
Unique file hashes: 11,469
Dataset 2 records: 8,810
BTXRD records: 3,746


## 4. Dataset 2 duplicate groups across splits

In [12]:
dataset2_records = hash_records[hash_records["dataset"] == "Dataset2"].copy()

grouped_dataset2 = dataset2_records.groupby("hash", sort=True)
cross_split_groups = []
cross_split_rows = []

for image_hash, group in grouped_dataset2:
    splits = tuple(sorted(group["split"].unique(), key=SPLITS.index))
    if len(splits) < 2:
        continue

    labels = tuple(sorted(group["cancer"].unique()))
    group_info = {
        "hash": image_hash,
        "copies": len(group),
        "splits": " + ".join(splits),
        "split_count": len(splits),
        "labels": ", ".join(map(str, labels)),
        "label_status": "identical" if len(labels) == 1 else "conflicting",
    }
    cross_split_groups.append(group_info)

    for row in group.sort_values(["split", "filename"], key=lambda column: column.map({split: index for index, split in enumerate(SPLITS)}) if column.name == "split" else column).itertuples(index=False):
        cross_split_rows.append({
            "hash": image_hash,
            "copies_in_group": len(group),
            "filename": row.filename,
            "dataset": row.dataset,
            "split": row.split,
            "cancer": row.cancer,
            "label_status": group_info["label_status"],
        })

duplicate_groups = pd.DataFrame(cross_split_groups)
duplicate_copies = pd.DataFrame(cross_split_rows)

print(f"Unique Dataset 2 cross-split duplicate groups: {len(duplicate_groups):,}")
print(f"Dataset 2 image copies in cross-split groups: {len(duplicate_copies):,}")
print("\nEvery duplicate group, one row per copy:")
display(duplicate_copies.sort_values(["hash", "split", "filename"]).reset_index(drop=True))

Unique Dataset 2 cross-split duplicate groups: 346
Dataset 2 image copies in cross-split groups: 692

Every duplicate group, one row per copy:


,hash,copies_in_group,filename,dataset,split,cancer,label_status
0,02210274f4097ccd35b8032b591db7546619e6e9671d7d...,2,Dataset2_forearm_8_2_png.rf.47cb418ae0a74d5082...,Dataset2,train,0,identical
1,02210274f4097ccd35b8032b591db7546619e6e9671d7d...,2,Dataset2_forearm_8_2_png.rf.dbd52c6defdaf47c73...,Dataset2,valid,0,identical
2,026b54b1f4df7292527a08c254325035cfb0913a437730...,2,Dataset2_image-no546-normal-_png.rf.5094cefa44...,Dataset2,train,0,identical
3,026b54b1f4df7292527a08c254325035cfb0913a437730...,2,Dataset2_image-no546-normal-_png.rf.785030e537...,Dataset2,valid,0,identical
4,02aadfb44b0b95489acea49b02f74573b30b52024a5826...,2,Dataset2_image-no119-normal-_png.rf.cba774fcc7...,Dataset2,test,0,identical
...,...,...,...,...,...,...,...
687,fae6d7439ea50709a0b7b711312ae8f2c9d32d26296da3...,2,Dataset2_hand_0_2_png.rf.42dff84e83277518adce5...,Dataset2,valid,0,identical
688,fc996052b2036dc1e5f7be18fe70aeb0b280a72e6518a2...,2,Dataset2_hand_25_png.rf.d8ef4de376901826a9e8bd...,Dataset2,train,0,identical
689,fc996052b2036dc1e5f7be18fe70aeb0b280a72e6518a2...,2,Dataset2_hand_25_png.rf.a8f8772b83e238b20ca0d4...,Dataset2,valid,0,identical
690,ff9625729172e84485bdccf4f41acc989600ae1854b76f...,2,Dataset2_download_png.rf.3b9ea76bbff42c4273913...,Dataset2,test,0,identical


## 5. Pair-specific counts

Each pair count below is a count of unique hash groups. A group present in all three splits is included in each relevant pair, so pair counts must not be added together.

In [13]:
PAIR_NAMES = {
    ("train", "valid"): "Train + Valid",
    ("train", "test"): "Train + Test",
    ("valid", "test"): "Valid + Test",
}

pair_summary_rows = []
for pair, pair_name in PAIR_NAMES.items():
    pair_groups = duplicate_groups[duplicate_groups["splits"].apply(
        lambda value: all(split in value.split(" + ") for split in pair)
    )]
    pair_hashes = set(pair_groups["hash"])
    pair_records = duplicate_copies[duplicate_copies["hash"].isin(pair_hashes)]
    pair_summary_rows.append({
        "Duplicate location": pair_name,
        "Duplicate groups": len(pair_groups),
        "Unique images involved": pair_records["filename"].nunique(),
    })

summary_table = pd.DataFrame(pair_summary_rows)
summary_table.loc[len(summary_table)] = {
    "Duplicate location": "Any cross-split duplication",
    "Duplicate groups": len(duplicate_groups),
    "Unique images involved": duplicate_copies["filename"].nunique(),
}
display(summary_table)

,Duplicate location,Duplicate groups,Unique images involved
0,Train + Valid,169,338
1,Train + Test,161,322
2,Valid + Test,16,32
3,Any cross-split duplication,346,692


## 6. Label consistency and three-way duplicates

In [14]:
identical_label_groups = duplicate_groups[duplicate_groups["label_status"] == "identical"].copy()
conflicting_label_groups = duplicate_groups[duplicate_groups["label_status"] == "conflicting"].copy()
three_way_groups = duplicate_groups[duplicate_groups["split_count"] == 3].copy()

print(f"Duplicate groups with identical labels: {len(identical_label_groups):,}")
print(f"Duplicate groups with conflicting labels: {len(conflicting_label_groups):,}")
print(f"Duplicate groups present in Train + Valid + Test: {len(three_way_groups):,}")

print("\nConflicting-label groups:")
display(conflicting_label_groups)

print("\nThree-way groups:")
display(three_way_groups)

Duplicate groups with identical labels: 346
Duplicate groups with conflicting labels: 0
Duplicate groups present in Train + Valid + Test: 0

Conflicting-label groups:


,hash,copies,splits,split_count,labels,label_status



Three-way groups:


,hash,copies,splits,split_count,labels,label_status


## 7. Exact duplicates between BTXRD and Dataset 2

This comparison uses the final PNG bytes in `data/processed/combined/`, not filenames.

In [15]:
btxrd_hashes = set(hash_records.loc[hash_records["dataset"] == "BTXRD", "hash"])
dataset2_hashes = set(hash_records.loc[hash_records["dataset"] == "Dataset2", "hash"])
cross_dataset_hashes = btxrd_hashes & dataset2_hashes

print(f"Exact BTXRD/Dataset 2 duplicate hash groups: {len(cross_dataset_hashes):,}")
if cross_dataset_hashes:
    display(hash_records[hash_records["hash"].isin(cross_dataset_hashes)].sort_values("hash"))
else:
    print("No exact duplicates found between BTXRD and Dataset 2.")

Exact BTXRD/Dataset 2 duplicate hash groups: 0
No exact duplicates found between BTXRD and Dataset 2.


## 8. BTXRD cross-split confirmation

In [16]:
btxrd_records = hash_records[hash_records["dataset"] == "BTXRD"]
btxrd_cross_split_groups = []

for image_hash, group in btxrd_records.groupby("hash"):
    splits = tuple(group["split"].unique())
    if len(splits) > 1:
        btxrd_cross_split_groups.append({
            "hash": image_hash,
            "copies": len(group),
            "splits": " + ".join(splits),
        })

btxrd_cross_split_table = pd.DataFrame(btxrd_cross_split_groups)
print(f"BTXRD cross-split duplicate groups: {len(btxrd_cross_split_table):,}")
display(btxrd_cross_split_table if not btxrd_cross_split_table.empty else pd.DataFrame(columns=["hash", "copies", "splits"]))

BTXRD cross-split duplicate groups: 7


,hash,copies,splits
0,0036aa063ba353103f774526fd57dc030e445c0f705939...,2,train + test
1,6012bdef41f3f22b37796b2198836ed24aa6c187f425ef...,2,train + valid
2,6e0843f39ca4730294511669b8d3603f31464716923975...,2,train + valid
3,7a12a652035998cbe1275af61c1208f47ee026d89dbf9c...,2,train + test
4,963a009bce58443aa3b698a528b4dc02ebff96c85eb547...,2,train + test
5,9770cbd997e024bc0df2e9d925c271f5c0c93a790a7e3d...,2,train + valid
6,e9f5a675fb182e1496f5a0dc4cad00695eb1c3d2b2456c...,2,train + valid


## 9. Investigation summary and next step

The results above distinguish duplicate hash groups from image copies and identify whether labels agree. The next decision should be based on the duplicate groups and their provenance, not on randomly deleting files or choosing a split arbitrarily.

Recommended next investigation: trace duplicated Dataset 2 images back to their original source records, determine whether the split assignment is intentional or an upstream data error, and compare evaluation strategies before rebuilding any dataset.

DUPLICATE ANALYSIS COMPLETE — NO DATA MODIFIED